# Train Commutative Transformer

This notebook intentionally keeps orchestration thin. Split preparation, fitting, reporting, plotting, and persistence are delegated to shared utilities in `src.ml`.


In [ ]:
%load_ext autoreload
%autoreload 2

from dataclasses import asdict
from pathlib import Path

import pandas as pd

from src.ml import (
    LossWeightConfig,
    OptimizationConfig,
    display_experiment_summary,
    display_holdout_evaluation,
    create_experiment_run,
    fit_estimator_on_experiment,
    persist_experiment_artifacts,
    plot_holdout_branch_embedding_projections,
    plot_training_history,
    prepare_multitask_experiment_data,
)
from src.dataset_config import load_current_dataset_artifact_path
from src.tensor_utils import build_tensor_embedding_2d, load_labeled_tensor_dataset, plot_tensor_embedding_2d

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)


In [ ]:
from src.ml import CommutativeTransformerClassifier, CommutativeTransformerConfig


In [ ]:
# User inputs

dataset_artifact_path = load_current_dataset_artifact_path()
print(dataset_artifact_path.name)
experiment_output_dir = Path("artifacts/nb8_commutative_transformer")
experiment_run = create_experiment_run(experiment_output_dir, "8T_train_commutative_transformer")
loss_plot_dir = Path(experiment_run.loss_plot_dir) / "training"
figure_dir = Path(experiment_run.figure_dir)
persist_artifacts = True
print(f"Experiment id: {experiment_run.experiment_id}")
print(f"Experiment run folder: {Path(experiment_run.run_dir).resolve()}")
print(f"Training loss PDFs: {loss_plot_dir.resolve()}")

holdout_fraction = 0.25
validation_fraction_within_train = 0.20
train_num_random_rotations = 6
rotation_range_degrees = 12.0

model_config = CommutativeTransformerConfig(
    spatial_patch_size_st=(1, 16, 16),
    spatial_patch_size_ts=(1, 16, 16),
    temporal_patch_size_ts=1,
    embed_dim=96,
    num_heads=4,
    mlp_ratio=4.0,
    dropout=0.2,
    attention_dropout=0.0,
    st_spatial_depth=2,
    st_temporal_depth=2,
    ts_temporal_depth=2,
    ts_spatial_depth=2,
    embedding_dim=96,
    probe_region_grid=(1, 2, 2),
    probe_time_bins=8,
    probe_frequency_bins=4,
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=100,
    learning_rate=1e-4,
    weight_decay=1e-3,
    early_stopping_patience=12,
    early_stopping_min_delta=1e-3,
    training_plot_dir=str(loss_plot_dir),
    training_plot_every_n_epochs=1,
    scheduler_patience=4,
    scheduler_factor=0.7,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    action_weight=1.0,
    compound_weight=0.2,
    concentration_weight=0.2,
    lambda_align=0.0,
)


In [ ]:
dataset = load_labeled_tensor_dataset(dataset_artifact_path)


In [ ]:
experiment = prepare_multitask_experiment_data(
    dataset,
    holdout_fraction=holdout_fraction,
    validation_fraction_within_train=validation_fraction_within_train,
    train_num_random_rotations=train_num_random_rotations,
    rotation_range_degrees=rotation_range_degrees,
    random_state=optimization_config.random_state,
)


In [ ]:
display_experiment_summary(experiment)


In [ ]:
model = CommutativeTransformerClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)


In [ ]:
fit_estimator_on_experiment(model, experiment)


In [ ]:
print(f"Writing training loss PDFs to: {loss_plot_dir.resolve()}")
plot_training_history(model, title="Commutative Transformer loss curves", loess_frac=0.6);


In [ ]:
holdout_evaluation = display_holdout_evaluation(model, experiment)


In [ ]:
holdout_embedding_projection = build_tensor_embedding_2d(
    model.transform(experiment.splits.X_holdout),
    experiment.y_true_holdout["action"],
    label_map=experiment.label_maps["action"],
    metadata=experiment.splits.metadata_holdout,
    method="umap",
    random_state=optimization_config.random_state,
)
holdout_embedding_projection.to_csv(
    figure_dir / f"{experiment_run.experiment_id}_holdout_embedding_umap.csv",
    index=False,
)
plot_tensor_embedding_2d(
    holdout_embedding_projection,
    title="Holdout embedding projection by action",
    marker_column="compound",
    output_path=figure_dir / f"{experiment_run.experiment_id}_holdout_embedding_umap.pdf",
)

In [ ]:
run_config = {
    "experiment_id": experiment_run.experiment_id,
    "experiment_run_dir": Path(experiment_run.run_dir),
    "dataset_artifact_path": dataset_artifact_path,
    "holdout_fraction": holdout_fraction,
    "validation_fraction_within_train": validation_fraction_within_train,
    "train_num_random_rotations": train_num_random_rotations,
    "rotation_range_degrees": rotation_range_degrees,
    "model_config": asdict(model_config),
    "optimization_config": asdict(optimization_config),
    "loss_weight_config": asdict(loss_weight_config),
}

In [ ]:
if persist_artifacts:
    experiment_artifacts = persist_experiment_artifacts(
        output_dir=experiment_output_dir,
        estimator=model,
        reports=holdout_evaluation.reports,
        config=run_config,
        experiment_prefix="8T_train_commutative_transformer",
        experiment_id=experiment_run.experiment_id,
        evaluation=holdout_evaluation,
        experiment=experiment,
        loss_plot_dirs=[loss_plot_dir],
    )
    experiment_artifacts